In [3]:
# =====================================================================
# IMPORTS
# =====================================================================
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr
import warnings
import os

warnings.filterwarnings("ignore")

# =====================================================================
# 1. KAGGLE T4x2 DUAL GPU SETUP
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Using device: {device} with {n_gpus} GPU(s)")

# =====================================================================
# 2. DATA PREPARATION (KAGGLE PATHS)
# =====================================================================
def load_and_prep_data():
    path_train = "/kaggle/input/datasets/ericdu847/v5g0-vcc-00-probe/adata_Training.h5ad"
    path_replogle = "/kaggle/input/datasets/ericdu847/v5g0-vcc-00-probe/replogle_2022_k562_gwps.h5ad"
    
    print("Loading datasets...")
    if not os.path.exists(path_train):
        raise FileNotFoundError(f"Cannot find {path_train}. Check Kaggle input paths.")
        
    adata = sc.read_h5ad(path_train)
    
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    train_adata = adata[adata.obs['target_gene'] == 'non-targeting'].copy()
    test_adata = adata[adata.obs['target_gene'] != 'non-targeting'].copy()
    
    print("Calculating Top 2500 Highly Variable Genes...")
    sc.pp.highly_variable_genes(train_adata, n_top_genes=2500)
    highly_variable_mask = train_adata.var.highly_variable
    
    train_adata = train_adata[:, highly_variable_mask]
    test_adata = test_adata[:, highly_variable_mask]
    
    X_train = train_adata.X.toarray() if hasattr(train_adata.X, "toarray") else train_adata.X
    X_test = test_adata.X.toarray() if hasattr(test_adata.X, "toarray") else test_adata.X
    labels_test = test_adata.obs['target_gene'].values
    
    gene_to_idx = {gene: idx for idx, gene in enumerate(test_adata.var_names)}
    
    return X_train, X_test, labels_test, gene_to_idx

# =====================================================================
# 3. CONDITIONAL AUTOENCODER 
# =====================================================================
class ConditionalAutoencoder(nn.Module):
    def __init__(self, n_genes, embed_dim=64):
        super(ConditionalAutoencoder, self).__init__()
        
        self.target_embedding = nn.Embedding(n_genes, embed_dim)
        
        self.encoder = nn.Sequential(
            nn.Linear(n_genes, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU()
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(256 + embed_dim, 512), nn.ReLU(),
            nn.Linear(512, n_genes)
        )

    def forward(self, x, target_idx):
        encoded = self.encoder(x)
        condition = self.target_embedding(target_idx)
        combined = torch.cat([encoded, condition], dim=1)
        return self.decoder(combined)

def train_conditional_ae(X_train, n_genes, epochs=15):
    print(f"\nTraining Conditional Autoencoder for {epochs} epochs on {n_genes} genes...")
    
    model = ConditionalAutoencoder(n_genes)
    
    # Wrap model for Multi-GPU processing on Kaggle's T4x2
    if n_gpus > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Increased batch size to 512 to fully utilize dual GPUs
    dataset = TensorDataset(torch.FloatTensor(X_train))
    dataloader = DataLoader(dataset, batch_size=512, shuffle=True)
    
    model.train()
    for epoch in range(epochs):
        for batch_idx, (batch_x,) in enumerate(dataloader):
            batch_size = batch_x.size(0)
            
            random_targets = torch.randint(0, n_genes, (batch_size,))
            
            masked_x = batch_x.clone()
            masked_x[torch.arange(batch_size), random_targets] = 0.0
            
            # Move data to GPU(s)
            masked_x = masked_x.to(device)
            batch_x = batch_x.to(device)
            random_targets = random_targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(masked_x, random_targets)
            loss = criterion(outputs, batch_x)
            loss.backward()
            optimizer.step()
            
            cells_processed = min((batch_idx + 1) * batch_size, len(dataset))
            print(f"Epoch: {epoch + 1}/{epochs} | Cells processed: {cells_processed}/{len(dataset)}".ljust(60), end='\r')
        print() 
    
    return model

def predict_conditional_ae(model, X_baseline_mean, target_gene_idx):
    model.eval()
    
    test_tensor = torch.FloatTensor(X_baseline_mean).unsqueeze(0).to(device)
    target_tensor = torch.tensor([target_gene_idx], dtype=torch.long).to(device)
    
    test_tensor[0, target_gene_idx] = 0.0 
    
    with torch.no_grad():
        pred_full = model(test_tensor, target_tensor).cpu().numpy()[0]
        
    pred_full[target_gene_idx] = 0.0
    return pred_full

# =====================================================================
# 4. EVALUATION
# =====================================================================
def evaluate_vcc_metrics(X_test_true, predictions_dict, labels_true, baseline_mean):
    print("\nEvaluating VCC Metrics...")
    target_genes = np.unique(labels_true)
    
    for model_name, X_test_pred in predictions_dict.items():
        results = []
        for target in target_genes:
            mask = (labels_true == target)
            
            true_pseudobulk = X_test_true[mask].mean(axis=0)
            pred_pseudobulk = X_test_pred[mask].mean(axis=0)
            
            mae_k = mean_absolute_error(true_pseudobulk, pred_pseudobulk)
            
            delta_true = true_pseudobulk - baseline_mean
            delta_pred = pred_pseudobulk - baseline_mean
            
            des_k, _ = pearsonr(delta_true, delta_pred)
            if np.isnan(des_k): des_k = 0.0
                
            results.append({
                'target': target, 'mae': mae_k, 'des': des_k,
                'delta_true': delta_true, 'delta_pred': delta_pred
            })
            
        df_results = pd.DataFrame(results)
        
        deltas_true_matrix = np.stack(df_results['delta_true'].values)
        deltas_pred_matrix = np.stack(df_results['delta_pred'].values)
        
        distances = cdist(deltas_pred_matrix, deltas_true_matrix, metric='cosine')
        pds_scores = []
        N = len(df_results)
        
        for i in range(N):
            rank = np.argsort(distances[i]).tolist().index(i)
            pds_k = 1.0 - (rank / max(1, N - 1))
            pds_scores.append(pds_k)
            
        df_results['pds'] = pds_scores
        
        print(f"\n--- {model_name.upper()} RESULTS ---")
        print(f"MAE: {df_results['mae'].mean():.4f}")
        print(f"DES: {df_results['des'].mean():.4f}")
        print(f"PDS: {df_results['pds'].mean():.4f}")

# =====================================================================
# MAIN EXECUTION
# =====================================================================
if __name__ == "__main__":
    X_train, X_test, labels_test, gene_to_idx = load_and_prep_data()
    n_genes = X_train.shape[1]
    X_baseline_mean = X_train.mean(axis=0)
    
    model = train_conditional_ae(X_train, n_genes, epochs=15)
    
    print("\nGenerating Predictions...")
    preds = np.zeros_like(X_test)
    
    for i, target_gene in enumerate(labels_test):
        target_idx = gene_to_idx.get(target_gene, 0) 
        preds[i] = predict_conditional_ae(model, X_baseline_mean, target_idx)

    evaluate_vcc_metrics(X_test, {"Explicit Conditional AE": preds}, labels_test, X_baseline_mean)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject